# 04 — Evaluation & Visualizations

Compare the three retrieval modes:
1. **Clinical only** (PubMedQA) — research-grade evidence
2. **Patient only** (MedQuAD) — plain language explanations
3. **Combined** — dual-corpus retrieval

Metrics: accuracy, macro-F1, per-class precision/recall, confusion matrices, embedding visualizations.

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.evaluation import (
    compute_classification_metrics,
    plot_confusion_matrix,
    plot_retrieval_recall,
    plot_embedding_clusters,
    plot_answer_comparison,
)

sns.set_style('whitegrid')

In [ ]:
# Load saved evaluation results
eval_results = {}
for mode in ['clinical', 'patient', 'combined']:
    with open(f'../data/eval_results_{mode}.json', 'r') as f:
        eval_results[mode] = json.load(f)
    print(f"{mode}: {len(eval_results[mode])} examples")

## 1. Classification Metrics (Yes/No/Maybe)

In [ ]:
# Compute metrics for each mode
metrics = {}
for mode, results in eval_results.items():
    y_true = [r['true_decision'] for r in results]
    y_pred = [r['decision'] for r in results]
    metrics[mode] = compute_classification_metrics(y_true, y_pred)

    print(f"\n{'='*60}")
    print(f"Mode: {mode.upper()}")
    print(f"{'='*60}")
    print(f"Accuracy: {metrics[mode]['accuracy']:.3f}")
    print(f"Macro F1: {metrics[mode]['macro_f1']:.3f}")
    print(f"\n{metrics[mode]['classification_report']}")

In [ ]:
# Side-by-side accuracy comparison
fig, ax = plt.subplots(figsize=(8, 5))

modes = list(metrics.keys())
accuracies = [metrics[m]['accuracy'] for m in modes]
f1_scores = [metrics[m]['macro_f1'] for m in modes]

x = np.arange(len(modes))
width = 0.35

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#2196F3')
bars2 = ax.bar(x + width/2, f1_scores, width, label='Macro F1', color='#FF9800')

ax.set_ylabel('Score')
ax.set_title('RAG Performance by Retrieval Mode')
ax.set_xticks(x)
ax.set_xticklabels([m.title() for m in modes])
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../figures/mode_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (mode, results) in enumerate(eval_results.items()):
    y_true = [r['true_decision'] for r in results]
    y_pred = [r['decision'] for r in results]
    
    from sklearn.metrics import confusion_matrix
    labels = ['yes', 'no', 'maybe']
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=axes[i])
    axes[i].set_title(f'{mode.title()} Retrieval')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True')

plt.suptitle('Confusion Matrices by Retrieval Mode', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Qualitative Comparison

In [ ]:
# Show side-by-side answers for a few examples
n_examples = 5
print("Qualitative Answer Comparison\n")

for i in range(min(n_examples, len(eval_results['clinical']))):
    print(f"{'='*80}")
    print(f"Q: {eval_results['clinical'][i]['question']}")
    print(f"True: {eval_results['clinical'][i]['true_decision']}")
    print()
    for mode in ['clinical', 'patient', 'combined']:
        r = eval_results[mode][i]
        print(f"  [{mode.upper():>8}] Decision: {r['decision']}")
        print(f"           Answer: {r['answer'][:200]}...")
        print()

In [ ]:
# Plot answer comparison table
comparison_examples = []
for i in range(min(8, len(eval_results['combined']))):
    comparison_examples.append({
        'question': eval_results['combined'][i]['question'],
        'true_decision': eval_results['combined'][i]['true_decision'],
        'pred_decision': eval_results['combined'][i]['decision'],
    })

plot_answer_comparison(comparison_examples, save_path='../figures/answer_comparison.png')

## 4. Embedding Visualization (UMAP)

In [ ]:
# Visualize embeddings from both corpora to show corpus separation
# (Run this only if retriever objects are still in memory, or reload)
# This cell assumes you have clinical_embeddings and patient_embeddings from notebook 03

try:
    import umap
    
    # Sample for visualization (UMAP is slow on large datasets)
    n_sample = 2000
    
    # If embeddings are available from notebook 03:
    # Otherwise, re-encode a sample
    from src.retriever import BiomedRetriever
    retriever = BiomedRetriever()
    
    # Load some samples from each corpus
    from src.data_utils import load_pubmedqa, filter_health_domain
    from datasets import load_dataset
    
    art_ds = load_pubmedqa('pqa_artificial')
    health_ex = filter_health_domain(art_ds['train'])
    clinical_sample = []
    for ex in health_ex[:500]:
        if isinstance(ex.get('context'), dict):
            for ctx in ex['context'].get('contexts', []):
                clinical_sample.append(ctx)
                if len(clinical_sample) >= n_sample // 2:
                    break
        if len(clinical_sample) >= n_sample // 2:
            break
    
    medquad = load_dataset('lavita/MedQuAD', split='train')
    patient_sample = [
        item['question'] + ' ' + item['answer']
        for item in medquad
        if item['answer']
    ][:n_sample // 2]
    
    all_texts = clinical_sample + patient_sample
    all_labels = ['Clinical'] * len(clinical_sample) + ['Patient'] * len(patient_sample)
    
    print(f"Encoding {len(all_texts)} texts for UMAP...")
    all_embeddings = retriever.encode(all_texts)
    
    plot_embedding_clusters(
        all_embeddings, all_labels,
        title='Document Embeddings: Clinical vs Patient Corpus (UMAP)',
        save_path='../figures/embedding_umap.png'
    )
except ImportError:
    print("Install umap-learn: pip install umap-learn")

## 5. Summary

| Mode | Accuracy | Macro F1 | Key Observation |
|------|----------|----------|----------------|
| Clinical only | _TBD_ | _TBD_ | Technical, evidence-heavy answers |
| Patient only | _TBD_ | _TBD_ | Readable, but less evidence-backed |
| Combined | _TBD_ | _TBD_ | Best of both: grounded + accessible |

### Key Findings
_(To be filled after running experiments)_

### Limitations & Responsible Use
- This system is for **educational/research purposes only** and should not be used for medical advice
- PubMedQA labels are expert-annotated but the question framing is clinical, not patient-facing
- 4-bit quantization may reduce generation quality compared to full precision
- Hallucination risk: LLM may generate plausible-sounding but unsupported claims
- For production deployment (e.g., Reen), answers should be reviewed by medical professionals